In [8]:
from langchain_community.retrievers import BM25Retriever
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from typing import List
from dotenv import load_dotenv

load_dotenv()

# Custom Ensemble Retriever
class EnsembleRetriever(BaseRetriever):
    """Combines BM25 (keyword) and semantic (embedding) retrievers"""
    
    def __init__(self, retrievers: List[BaseRetriever], weights: List[float]):
        super().__init__()
        self.retrievers = retrievers
        self.weights = weights
    
    def _get_relevant_documents(self, query: str) -> List[Document]:
        """Combine results from all retrievers"""
        all_docs = {}
        
        # Get results from each retriever
        for retriever, weight in zip(self.retrievers, self.weights):
            docs = retriever.invoke(query)
            for i, doc in enumerate(docs):
                # Use content as key to avoid duplicates
                key = doc.page_content
                if key not in all_docs:
                    all_docs[key] = {"doc": doc, "score": 0}
                # Higher rank (lower index) = higher score
                all_docs[key]["score"] += weight * (1 / (i + 1))
        
        # Sort by combined score
        sorted_docs = sorted(
            all_docs.items(),
            key=lambda x: x[1]["score"],
            reverse=True
        )
        
        return [doc_info["doc"] for _, doc_info in sorted_docs]

print("✓ Custom EnsembleRetriever defined!")

✓ Custom EnsembleRetriever defined!


In [2]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.
